# Практическая работа № 2. Создание, заполнение, загрузка и обработка хранилища данных

**ИТАиОД · вариант 7 · R**

Воспроизведём операции из методички, представим квартальный ряд `austres` в виде `data.frame` и обработаем ответы волны 2022 из [Stack Overflow Developer Survey 2011–2022](https://www.kaggle.com/datasets/yasirabdaali/stack-overflow-developer-survey-20112022). Закончим сравнением времени выполнения одной и той же операции в base R и `dplyr`.

Запустите все ячейки в ядре R. Нужны пакеты `dplyr` и `nycflights13`. При работе без интернета укажите путь к CSV 2022 года через `ITAIOD_SURVEY_CSV`.

В CSV за 2022 год из архива Kaggle обнаружен повреждённый участок. Для этой волны используется [оригинальная публикация Stack Overflow](https://survey.stackoverflow.co/2022/).

## 1. Таблицы в base R

Примеры 2.1–2.6. Создаём таблицу с числовым, текстовым и логическим столбцами, выбираем и фильтруем строки, объединяем таблицы по строкам, по столбцам и по ключу.

In [1]:
df <- data.frame(x = 1:4, y = LETTERS[1:4], z = c(TRUE, FALSE))
print(df)
str(df)
print(c(rows = nrow(df), columns = ncol(df)))
print(dim(df))

print(list(
  rows_3_and_4 = df[3:4, -1, drop = FALSE],
  first_column = df[, 1],
  odd_rows = df[c(TRUE, FALSE), c("z", "x")],
  column_z = df$z,
  column_x = df[["x"]]
))
print(df[df$x > 2, ])
print(subset(df, x > 2))
print(subset(df, x > 2, select = c(x, y)))

  x y     z
1 1 A  TRUE
2 2 B FALSE
3 3 C  TRUE
4 4 D FALSE


'data.frame':	4 obs. of  3 variables:
 $ x: int  1 2 3 4
 $ y: chr  "A" "B" "C" "D"
 $ z: logi  TRUE FALSE TRUE FALSE


   rows columns 
      4       3 


[1] 4 3


$rows_3_and_4
  y     z
3 C  TRUE
4 D FALSE

$first_column
[1] 1 2 3 4

$odd_rows
     z x
1 TRUE 1
3 TRUE 3

$column_z
[1]  TRUE FALSE  TRUE FALSE

$column_x
[1] 1 2 3 4



  x y     z
3 3 C  TRUE
4 4 D FALSE


  x y     z
3 3 C  TRUE
4 4 D FALSE


  x y
3 3 C
4 4 D


In [2]:
df_more <- data.frame(x = 5:8, y = LETTERS[5:8], z = c(TRUE, FALSE))
df_key <- data.frame(x = c(3, 2, 6, 1), s = c(100, 1000, 300, 500))

print(rbind(df, df_more))
print(cbind(df, df_more))
merge(df, df_key, by = "x")

  x y     z
1 1 A  TRUE
2 2 B FALSE
3 3 C  TRUE
4 4 D FALSE
5 5 E  TRUE
6 6 F FALSE
7 7 G  TRUE
8 8 H FALSE


  x y     z x y     z
1 1 A  TRUE 5 E  TRUE
2 2 B FALSE 6 F FALSE
3 3 C  TRUE 7 G  TRUE
4 4 D FALSE 8 H FALSE


x,y,z,s
<int>,<chr>,<lgl>,<dbl>
1,A,TRUE,500
2,B,FALSE,1000
3,C,TRUE,100


В примере 2.7 номинальные признаки `vs` и `am` становятся факторами, а количество цилиндров, передач и карбюраторов — упорядоченными факторами.

In [3]:
cars <- datasets::mtcars
print(typeof(cars))
print(colnames(cars))
print(rownames(cars))
str(cars)

cars_with_scales <- within(cars, {
  vs <- factor(vs, levels = c(0, 1), labels = c("V", "S"))
  am <- factor(am, levels = c(0, 1), labels = c("automatic", "manual"))
  cyl <- ordered(cyl)
  gear <- ordered(gear)
  carb <- ordered(carb)
})
str(cars_with_scales)
summary(cars_with_scales)

[1] "list"


 [1] "mpg"  "cyl"  "disp" "hp"   "drat" "wt"   "qsec" "vs"   "am"   "gear"
[11] "carb"


 [1] "Mazda RX4"           "Mazda RX4 Wag"       "Datsun 710"         
 [4] "Hornet 4 Drive"      "Hornet Sportabout"   "Valiant"            
 [7] "Duster 360"          "Merc 240D"           "Merc 230"           
[10] "Merc 280"            "Merc 280C"           "Merc 450SE"         
[13] "Merc 450SL"          "Merc 450SLC"         "Cadillac Fleetwood" 
[16] "Lincoln Continental" "Chrysler Imperial"   "Fiat 128"           
[19] "Honda Civic"         "Toyota Corolla"      "Toyota Corona"      
[22] "Dodge Challenger"    "AMC Javelin"         "Camaro Z28"         
[25] "Pontiac Firebird"    "Fiat X1-9"           "Porsche 914-2"      
[28] "Lotus Europa"        "Ford Pantera L"      "Ferrari Dino"       
[31] "Maserati Bora"       "Volvo 142E"         


'data.frame':	32 obs. of  11 variables:
 $ mpg : num  21 21 22.8 21.4 18.7 18.1 14.3 24.4 22.8 19.2 ...
 $ cyl : num  6 6 4 6 8 6 8 4 4 6 ...
 $ disp: num  160 160 108 258 360 ...
 $ hp  : num  110 110 93 110 175 105 245 62 95 123 ...
 $ drat: num  3.9 3.9 3.85 3.08 3.15 2.76 3.21 3.69 3.92 3.92 ...
 $ wt  : num  2.62 2.88 2.32 3.21 3.44 ...
 $ qsec: num  16.5 17 18.6 19.4 17 ...
 $ vs  : num  0 0 1 1 0 1 0 1 1 1 ...
 $ am  : num  1 1 1 0 0 0 0 0 0 0 ...
 $ gear: num  4 4 4 3 3 3 3 4 4 4 ...
 $ carb: num  4 4 1 1 2 1 4 2 2 4 ...


'data.frame':	32 obs. of  11 variables:
 $ mpg : num  21 21 22.8 21.4 18.7 18.1 14.3 24.4 22.8 19.2 ...
 $ cyl : Ord.factor w/ 3 levels "4"<"6"<"8": 2 2 1 2 3 2 3 1 1 2 ...
 $ disp: num  160 160 108 258 360 ...
 $ hp  : num  110 110 93 110 175 105 245 62 95 123 ...
 $ drat: num  3.9 3.9 3.85 3.08 3.15 2.76 3.21 3.69 3.92 3.92 ...
 $ wt  : num  2.62 2.88 2.32 3.21 3.44 ...
 $ qsec: num  16.5 17 18.6 19.4 17 ...
 $ vs  : Factor w/ 2 levels "V","S": 1 1 2 2 1 2 1 2 2 2 ...
 $ am  : Factor w/ 2 levels "automatic","manual": 2 2 2 1 1 1 1 1 1 1 ...
 $ gear: Ord.factor w/ 3 levels "3"<"4"<"5": 2 2 2 1 1 1 1 2 2 2 ...
 $ carb: Ord.factor w/ 6 levels "1"<"2"<"3"<"4"<..: 4 4 1 1 2 1 4 2 2 4 ...


      mpg        cyl         disp             hp             drat      
 Min.   :10.40   4:11   Min.   : 71.1   Min.   : 52.0   Min.   :2.760  
 1st Qu.:15.43   6: 7   1st Qu.:120.8   1st Qu.: 96.5   1st Qu.:3.080  
 Median :19.20   8:14   Median :196.3   Median :123.0   Median :3.695  
 Mean   :20.09          Mean   :230.7   Mean   :146.7   Mean   :3.597  
 3rd Qu.:22.80          3rd Qu.:326.0   3rd Qu.:180.0   3rd Qu.:3.920  
 Max.   :33.90          Max.   :472.0   Max.   :335.0   Max.   :4.930  
       wt             qsec       vs             am     gear   carb  
 Min.   :1.513   Min.   :14.50   V:18   automatic:19   3:15   1: 7  
 1st Qu.:2.581   1st Qu.:16.89   S:14   manual   :13   4:12   2:10  
 Median :3.325   Median :17.71                         5: 5   3: 3  
 Mean   :3.217   Mean   :17.85                                4:10  
 3rd Qu.:3.610   3rd Qu.:18.90                                6: 1  
 Max.   :5.424   Max.   :22.90                                8: 1  

## 2. Операции `dplyr` из методички

Используем `nycflights13::flights`. Для больших выборок показываем размер и первые строки.

In [4]:
suppressPackageStartupMessages(library(dplyr))

flights <- nycflights13::flights
print(dim(flights))
print(head(flights))

january_1 <- filter(flights, month == 1, day == 1)
january_february <- filter(flights, month == 1 | month == 2)
print(c(january_1 = nrow(january_1), january_february = nrow(january_february)))
print(head(january_1))
print(slice(flights, 1:10))
print(head(arrange(flights, year, month, day)))
head(arrange(flights, desc(arr_delay)))

[1] 336776     19


# A tibble: 6 × 19
   year month   day dep_time sched_dep_time dep_delay arr_time sched_arr_time
  <int> <int> <int>    <int>          <int>     <dbl>    <int>          <int>
1  2013     1     1      517            515         2      830            819
2  2013     1     1      533            529         4      850            830
3  2013     1     1      542            540         2      923            850
4  2013     1     1      544            545        -1     1004           1022
5  2013     1     1      554            600        -6      812            837
6  2013     1     1      554            558        -4      740            728
# ℹ 11 more variables: arr_delay <dbl>, carrier <chr>, flight <int>,
#   tailnum <chr>, origin <chr>, dest <chr>, air_time <dbl>, distance <dbl>,
#   hour <dbl>, minute <dbl>, time_hour <dttm>


       january_1 january_february 
             842            51955 


# A tibble: 6 × 19
   year month   day dep_time sched_dep_time dep_delay arr_time sched_arr_time
  <int> <int> <int>    <int>          <int>     <dbl>    <int>          <int>
1  2013     1     1      517            515         2      830            819
2  2013     1     1      533            529         4      850            830
3  2013     1     1      542            540         2      923            850
4  2013     1     1      544            545        -1     1004           1022
5  2013     1     1      554            600        -6      812            837
6  2013     1     1      554            558        -4      740            728
# ℹ 11 more variables: arr_delay <dbl>, carrier <chr>, flight <int>,
#   tailnum <chr>, origin <chr>, dest <chr>, air_time <dbl>, distance <dbl>,
#   hour <dbl>, minute <dbl>, time_hour <dttm>


# A tibble: 10 × 19
    year month   day dep_time sched_dep_time dep_delay arr_time sched_arr_time
   <int> <int> <int>    <int>          <int>     <dbl>    <int>          <int>
 1  2013     1     1      517            515         2      830            819
 2  2013     1     1      533            529         4      850            830
 3  2013     1     1      542            540         2      923            850
 4  2013     1     1      544            545        -1     1004           1022
 5  2013     1     1      554            600        -6      812            837
 6  2013     1     1      554            558        -4      740            728
 7  2013     1     1      555            600        -5      913            854
 8  2013     1     1      557            600        -3      709            723
 9  2013     1     1      557            600        -3      838            846
10  2013     1     1      558            600        -2      753            745
# ℹ 11 more variables: arr_delay

# A tibble: 6 × 19
   year month   day dep_time sched_dep_time dep_delay arr_time sched_arr_time
  <int> <int> <int>    <int>          <int>     <dbl>    <int>          <int>
1  2013     1     1      517            515         2      830            819
2  2013     1     1      533            529         4      850            830
3  2013     1     1      542            540         2      923            850
4  2013     1     1      544            545        -1     1004           1022
5  2013     1     1      554            600        -6      812            837
6  2013     1     1      554            558        -4      740            728
# ℹ 11 more variables: arr_delay <dbl>, carrier <chr>, flight <int>,
#   tailnum <chr>, origin <chr>, dest <chr>, air_time <dbl>, distance <dbl>,
#   hour <dbl>, minute <dbl>, time_hour <dttm>


year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
<int>,<int>,<int>,<int>,<int>,<dbl>,<int>,<int>,<dbl>,<chr>,<int>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dttm>
2013,1,9,641,900,1301,1242,1530,1272,HA,51,N384HA,JFK,HNL,640,4983,9,0,2013-01-09 09:00:00
2013,6,15,1432,1935,1137,1607,2120,1127,MQ,3535,N504MQ,JFK,CMH,74,483,19,35,2013-06-15 19:00:00
2013,1,10,1121,1635,1126,1239,1810,1109,MQ,3695,N517MQ,EWR,ORD,111,719,16,35,2013-01-10 16:00:00
2013,9,20,1139,1845,1014,1457,2210,1007,AA,177,N338AA,JFK,SFO,354,2586,18,45,2013-09-20 18:00:00
2013,7,22,845,1600,1005,1044,1815,989,MQ,3075,N665MQ,JFK,CVG,96,589,16,0,2013-07-22 16:00:00
2013,4,10,1100,1900,960,1342,2211,931,DL,2391,N959DL,JFK,TPA,139,1005,19,0,2013-04-10 19:00:00


In [5]:
print(head(select(flights, year, month, day)))
print(head(select(flights, year:day)))
print(head(select(flights, -(year:day))))
print(head(select(flights, starts_with("dep_"), ends_with("_delay"))))
print(head(select(flights, tail_num = tailnum)))
print(head(rename(flights, tail_num = tailnum)))
print(head(distinct(select(flights, tailnum))))
head(distinct(select(flights, origin, dest)))

# A tibble: 6 × 3
   year month   day
  <int> <int> <int>
1  2013     1     1
2  2013     1     1
3  2013     1     1
4  2013     1     1
5  2013     1     1
6  2013     1     1


# A tibble: 6 × 3
   year month   day
  <int> <int> <int>
1  2013     1     1
2  2013     1     1
3  2013     1     1
4  2013     1     1
5  2013     1     1
6  2013     1     1


# A tibble: 6 × 16
  dep_time sched_dep_time dep_delay arr_time sched_arr_time arr_delay carrier
     <int>          <int>     <dbl>    <int>          <int>     <dbl> <chr>  
1      517            515         2      830            819        11 UA     
2      533            529         4      850            830        20 UA     
3      542            540         2      923            850        33 AA     
4      544            545        -1     1004           1022       -18 B6     
5      554            600        -6      812            837       -25 DL     
6      554            558        -4      740            728        12 UA     
# ℹ 9 more variables: flight <int>, tailnum <chr>, origin <chr>, dest <chr>,
#   air_time <dbl>, distance <dbl>, hour <dbl>, minute <dbl>, time_hour <dttm>


# A tibble: 6 × 3
  dep_time dep_delay arr_delay
     <int>     <dbl>     <dbl>
1      517         2        11
2      533         4        20
3      542         2        33
4      544        -1       -18
5      554        -6       -25
6      554        -4        12


# A tibble: 6 × 1
  tail_num
  <chr>   
1 N14228  
2 N24211  
3 N619AA  
4 N804JB  
5 N668DN  
6 N39463  


# A tibble: 6 × 19
   year month   day dep_time sched_dep_time dep_delay arr_time sched_arr_time
  <int> <int> <int>    <int>          <int>     <dbl>    <int>          <int>
1  2013     1     1      517            515         2      830            819
2  2013     1     1      533            529         4      850            830
3  2013     1     1      542            540         2      923            850
4  2013     1     1      544            545        -1     1004           1022
5  2013     1     1      554            600        -6      812            837
6  2013     1     1      554            558        -4      740            728
# ℹ 11 more variables: arr_delay <dbl>, carrier <chr>, flight <int>,
#   tail_num <chr>, origin <chr>, dest <chr>, air_time <dbl>, distance <dbl>,
#   hour <dbl>, minute <dbl>, time_hour <dttm>


# A tibble: 6 × 1
  tailnum
  <chr>  
1 N14228 
2 N24211 
3 N619AA 
4 N804JB 
5 N668DN 
6 N39463 


origin,dest
<chr>,<chr>
EWR,IAH
LGA,IAH
JFK,MIA
JFK,BQN
LGA,ATL
EWR,ORD


В примере `transform()` исправляем имя входного столбца на `dep_delay`. Затем перехватываем ожидаемую ошибку: `transform()` вычисляет выражения в исходном контексте, где ещё нет столбца `gain`. В `mutate()` вычисления внутри вызова идут последовательно.

In [6]:
print(head(mutate(flights,
  gain = arr_delay - dep_delay,
  speed = distance / air_time * 60
)))
print(head(mutate(flights,
  gain = arr_delay - dep_delay,
  gain_per_hour = gain / (air_time / 60)
)))

transform_message <- tryCatch(
  transform(flights,
    gain = arr_delay - dep_delay,
    gain_per_hour = gain / (air_time / 60)
  ),
  error = function(error) conditionMessage(error)
)
print(transform_message)
print(head(transmute(flights,
  gain = arr_delay - dep_delay,
  gain_per_hour = gain / (air_time / 60)
)))
print(summarise(flights, delay = mean(dep_delay, na.rm = TRUE)))

set.seed(7)
print(sample_n(flights, 10))
sample_one_percent <- sample_frac(flights, 0.01)
print(nrow(sample_one_percent))
head(sample_one_percent)

# A tibble: 6 × 21
   year month   day dep_time sched_dep_time dep_delay arr_time sched_arr_time
  <int> <int> <int>    <int>          <int>     <dbl>    <int>          <int>
1  2013     1     1      517            515         2      830            819
2  2013     1     1      533            529         4      850            830
3  2013     1     1      542            540         2      923            850
4  2013     1     1      544            545        -1     1004           1022
5  2013     1     1      554            600        -6      812            837
6  2013     1     1      554            558        -4      740            728
# ℹ 13 more variables: arr_delay <dbl>, carrier <chr>, flight <int>,
#   tailnum <chr>, origin <chr>, dest <chr>, air_time <dbl>, distance <dbl>,
#   hour <dbl>, minute <dbl>, time_hour <dttm>, gain <dbl>, speed <dbl>


# A tibble: 6 × 21
   year month   day dep_time sched_dep_time dep_delay arr_time sched_arr_time
  <int> <int> <int>    <int>          <int>     <dbl>    <int>          <int>
1  2013     1     1      517            515         2      830            819
2  2013     1     1      533            529         4      850            830
3  2013     1     1      542            540         2      923            850
4  2013     1     1      544            545        -1     1004           1022
5  2013     1     1      554            600        -6      812            837
6  2013     1     1      554            558        -4      740            728
# ℹ 13 more variables: arr_delay <dbl>, carrier <chr>, flight <int>,
#   tailnum <chr>, origin <chr>, dest <chr>, air_time <dbl>, distance <dbl>,
#   hour <dbl>, minute <dbl>, time_hour <dttm>, gain <dbl>, gain_per_hour <dbl>


[1] "object 'gain' not found"


# A tibble: 6 × 2
   gain gain_per_hour
  <dbl>         <dbl>
1     9          2.38
2    16          4.23
3    31         11.6 
4   -17         -5.57
5   -19         -9.83
6    16          6.4 


# A tibble: 1 × 1
  delay
  <dbl>
1  12.6


# A tibble: 10 × 19
    year month   day dep_time sched_dep_time dep_delay arr_time sched_arr_time
   <int> <int> <int>    <int>          <int>     <dbl>    <int>          <int>
 1  2013    12    10      627            635        -8      934            930
 2  2013     9    27      624            635       -11      810            833
 3  2013    12     4      805            805         0     1040           1050
 4  2013     3    16      754            802        -8      907            919
 5  2013     7    31     1500           1459         1     1735           1737
 6  2013     3     8     1545           1330       135     1822           1640
 7  2013     2    12     1039           1039         0     1321           1329
 8  2013     2     7     2127           2125         2     2244           2250
 9  2013     3    22      849            900       -11      953           1022
10  2013     6     4     1153           1200        -7     1325           1314
# ℹ 11 more variables: arr_delay

[1] 3368


year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
<int>,<int>,<int>,<int>,<int>,<dbl>,<int>,<int>,<dbl>,<chr>,<int>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dttm>
2013,3,18,512,515,-3,749,814,-25,UA,1545,N37287,EWR,IAH,200,1400,5,15,2013-03-18 05:00:00
2013,6,30,1804,1710,54,2144,2015,89,AA,181,N338AA,JFK,LAX,320,2475,17,10,2013-06-30 17:00:00
2013,5,2,610,610,0,909,906,3,B6,315,N594JB,JFK,FLL,152,1069,6,10,2013-05-02 06:00:00
2013,9,1,853,900,-7,1053,1119,-26,DL,485,N320NB,EWR,ATL,100,746,9,0,2013-09-01 09:00:00
2013,10,16,801,805,-4,1030,1022,8,EV,4364,N11199,EWR,MCI,180,1092,8,5,2013-10-16 08:00:00
2013,7,26,947,959,-12,1102,1114,-12,EV,5711,N828AS,JFK,IAD,45,228,9,59,2013-07-26 09:00:00


## 3. Встроенные данные варианта

[`austres`](https://stat.ethz.ch/R-manual/R-devel/library/datasets/html/austres.html) — квартальный временной ряд численности жителей Австралии. В объекте R находятся 89 наблюдений со II квартала 1971 по II квартал 1993 года, что видно по `start()` и `end()`. Переводим ряд в таблицу, сохраняя год, квартал и исходные единицы измерения.

| Поле | Смысл | Тип | Единица |
|---|---|---|---|
| `year` | Календарный год | Целое, дискретное | Год |
| `quarter` | Номер квартала внутри года | Целое, порядковое | Квартал |
| `population_thousands` | Число жителей на дату наблюдения | Вещественное, количественное | Тысячи человек |

In [7]:
population_ts <- datasets::austres
population <- data.frame(
  year = as.integer(time(population_ts)),
  quarter = as.integer(cycle(population_ts)),
  population_thousands = as.numeric(population_ts)
)
print(head(population))
str(population)
print(summary(population))
print(rbind(start = start(population_ts), end = end(population_ts)))

population$quarter_change_thousands <- c(NA_real_, diff(population$population_thousands))
quarters_by_year <- table(population$year)
full_years <- as.integer(names(quarters_by_year[quarters_by_year == 4L]))
complete_years <- subset(population, year %in% full_years)
december <- subset(complete_years, quarter == 4L)

yearly_average <- aggregate(
  population_thousands ~ year, data = complete_years, FUN = mean
)
names(yearly_average)[2] <- "mean_population_thousands"
yearly <- merge(
  december[c("year", "population_thousands", "quarter_change_thousands")],
  yearly_average,
  by = "year"
)
print(head(yearly))
tail(yearly)

  year quarter population_thousands
1 1971       2              13067.3
2 1971       3              13130.5
3 1971       4              13198.4
4 1972       1              13254.2
5 1972       2              13303.7
6 1972       3              13353.9


'data.frame':	89 obs. of  3 variables:
 $ year                : int  1971 1971 1971 1972 1972 1972 1972 1973 1973 1973 ...
 $ quarter             : int  2 3 4 1 2 3 4 1 2 3 ...
 $ population_thousands: num  13067 13130 13198 13254 13304 ...


      year         quarter      population_thousands
 Min.   :1971   Min.   :1.000   Min.   :13067       
 1st Qu.:1976   1st Qu.:2.000   1st Qu.:14110       
 Median :1982   Median :2.000   Median :15184       
 Mean   :1982   Mean   :2.494   Mean   :15273       
 3rd Qu.:1987   3rd Qu.:3.000   3rd Qu.:16399       
 Max.   :1993   Max.   :4.000   Max.   :17662       


      [,1] [,2]
start 1971    2
end   1993    2


  year population_thousands quarter_change_thousands mean_population_thousands
1 1972              13409.3                     55.4                  13330.27
2 1973              13614.3                     61.7                  13532.65
3 1974              13832.0                     59.9                  13749.05
4 1975              13968.9                     42.1                  13912.83
5 1976              14110.1                     44.1                  14053.48
6 1977              14281.5                     49.8                  14215.25


,year,population_thousands,quarter_change_thousands,mean_population_thousands
,<int>,<dbl>,<dbl>,<dbl>
16,1987,16398.9,71.0,16298.27
17,1988,16697.0,75.4,16583.78
18,1989,16956.8,65.2,16864.67
19,1990,17169.4,62.5,17097.00
20,1991,17414.2,60.0,17324.95
21,1992,17568.7,42.7,17506.15


## 4. Внешние данные варианта

Одна строка — одна анкета. Для сравнения операций используем одну годовую волну, чтобы поля и их определения были одинаковыми для всех строк. Пропуски остаются `NA`.

| Поле | Смысл | Тип | Единица |
|---|---|---|---|
| `ResponseId` | Идентификатор анкеты в волне 2022 | Целое, номинальное | Безразмерный идентификатор |
| `Country` | Страна проживания по ответу участника | Строка, номинальное | Категория |
| `WorkExp` | Стаж работы по ответу участника | Число, количественное | Годы |
| `LanguageHaveWorkedWith` | Языки, указанные за последний год | Строка с несколькими значениями через `;` | Категории |

In [8]:
# Читает локальную или опубликованную таблицу ответов за 2022 год.
# Возвращает таблицу и имя фактически использованного источника.
read_survey_2022 <- function() {
  csv_paths <- c(
    Sys.getenv("ITAIOD_SURVEY_CSV"),
    "2022_survey_results_public.csv", "survey_results_public.csv",
    "../data/2022_survey_results_public.csv", "../data/survey_results_public.csv"
  )
  csv_paths <- csv_paths[file.exists(csv_paths)]
  if (length(csv_paths) > 0L) {
    path <- csv_paths[[1L]]
    return(list(
      data = read.csv(path, na.strings = c("", "NA"), fileEncoding = "UTF-8"),
      source = path
    ))
  }

  official_url <- paste0(
    "https://media.githubusercontent.com/media/StackExchange/Survey/",
    "main/packages/archive/2022/results.csv"
  )
  path <- tempfile(fileext = ".csv")
  on.exit(unlink(path))
  download.file(official_url, path, mode = "wb", quiet = TRUE)
  list(
    data = read.csv(path, na.strings = c("", "NA"), fileEncoding = "UTF-8"),
    source = official_url
  )
}

survey_import <- read_survey_2022()
survey <- survey_import$data
survey_columns <- c("ResponseId", "Country", "WorkExp", "LanguageHaveWorkedWith")
stopifnot(
  all(survey_columns %in% names(survey)),
  nrow(survey) == 73268L,
  anyDuplicated(survey$ResponseId) == 0L,
  is.numeric(survey$WorkExp)
)
cat("Источник:", survey_import$source, "\n")
cat("Ответов:", nrow(survey), "\n")

Источник: D:\sem7\tmp\itaiod-runtime\results.csv 


Ответов: 73268 


In [9]:
survey_fields <- survey |>
  select(ResponseId, Country, WorkExp, LanguageHaveWorkedWith) |>
  rename(country = Country, work_years = WorkExp) |>
  mutate(language_count = if_else(
    is.na(LanguageHaveWorkedWith),
    NA_integer_,
    lengths(strsplit(LanguageHaveWorkedWith, ";", fixed = TRUE))
  ))

print(slice(survey_fields, 1:5))
print(head(distinct(survey_fields, country)))
print(summarise(survey_fields,
  responses = n(),
  missing_country = sum(is.na(country)),
  missing_work_years = sum(is.na(work_years)),
  mean_work_years = mean(work_years, na.rm = TRUE)
))

experienced_by_country <- survey_fields |>
  filter(!is.na(country), work_years >= 5) |>
  group_by(country) |>
  summarise(
    responses = n(),
    mean_work_years = mean(work_years),
    .groups = "drop"
  ) |>
  arrange(desc(responses), country)
head(experienced_by_country, 10)

  ResponseId                                              country work_years
1          1                                                 <NA>         NA
2          2                                               Canada         NA
3          3 United Kingdom of Great Britain and Northern Ireland         NA
4          4                                               Israel         NA
5          5                             United States of America         NA
                       LanguageHaveWorkedWith language_count
1                                        <NA>             NA
2                       JavaScript;TypeScript              2
3           C#;C++;HTML/CSS;JavaScript;Python              5
4                C#;JavaScript;SQL;TypeScript              4
5 C#;HTML/CSS;JavaScript;SQL;Swift;TypeScript              6


                                               country
1                                                 <NA>
2                                               Canada
3 United Kingdom of Great Britain and Northern Ireland
4                                               Israel
5                             United States of America
6                                              Germany


  responses missing_country missing_work_years mean_work_years
1     73268            1497              36499        10.24238


country,responses,mean_work_years
<chr>,<int>,<dbl>
United States of America,5877,16.451251
Germany,1961,12.790413
India,1882,9.261424
United Kingdom of Great Britain and Northern Ireland,1645,15.017021
Canada,965,14.815544
Brazil,936,12.732906
France,694,12.324207
Poland,654,11.088685
Spain,628,13.547771


## 5. Сравнение скорости

В обоих случаях оставляем те же анкеты и считаем средний стаж по странам. Сначала проверяем равенство результатов с учётом числовой точности. Затем делаем по пять замеров, по 20 вызовов в каждом. Время чтения CSV и печати результатов в замер не входит.

In [10]:
# Средний стаж по странам среди респондентов со стажем от пяти лет.
# Обе функции получают исходную таблицу и возвращают одинаковую схему.
summarise_base <- function(data) {
  selected <- subset(data, !is.na(Country) & WorkExp >= 5)
  aggregate(WorkExp ~ Country, data = selected, FUN = mean)
}

summarise_dplyr <- function(data) {
  data |>
    filter(!is.na(Country), WorkExp >= 5) |>
    group_by(Country) |>
    summarise(WorkExp = mean(WorkExp), .groups = "drop")
}

base_result <- summarise_base(survey)
dplyr_result <- summarise_dplyr(survey)
dplyr_aligned <- dplyr_result[match(base_result$Country, dplyr_result$Country), ]
stopifnot(
  nrow(base_result) == nrow(dplyr_result),
  identical(base_result$Country, dplyr_aligned$Country),
  isTRUE(all.equal(base_result$WorkExp, dplyr_aligned$WorkExp))
)

# Возвращает миллисекунды на вызов в каждой серии измерений.
measure_ms <- function(method, data, batches = 5L, repetitions = 20L) {
  replicate(batches, {
    elapsed <- system.time(
      for (i in seq_len(repetitions)) invisible(method(data))
    )[["elapsed"]]
    elapsed * 1000 / repetitions
  })
}

base_ms <- measure_ms(summarise_base, survey)
dplyr_ms <- measure_ms(summarise_dplyr, survey)
timing <- data.frame(
  method = c("base R", "dplyr"),
  median_ms = c(median(base_ms), median(dplyr_ms)),
  min_ms = c(min(base_ms), min(dplyr_ms)),
  max_ms = c(max(base_ms), max(dplyr_ms))
)
timing

method,median_ms,min_ms,max_ms
<chr>,<dbl>,<dbl>,<dbl>
base R,61.5,49.5,73.5
dplyr,28.5,22.5,32.0


In [11]:
cat(sprintf(
  "Ряд austres: %d квартала; от %.1f до %.1f тыс. жителей.\n",
  nrow(population), population$population_thousands[1L],
  tail(population$population_thousands, 1L)
))
cat(sprintf(
  "Группировка по %d странам дала одинаковые средние в base R и dplyr.\n",
  nrow(base_result)
))
cat(sprintf(
  "Медиана замеров: %.3f мс (base R) и %.3f мс (dplyr) на вызов.\n",
  timing$median_ms[1L], timing$median_ms[2L]
))

Ряд austres: 89 квартала; от 13067.3 до 17661.5 тыс. жителей.


Группировка по 162 странам дала одинаковые средние в base R и dplyr.


Медиана замеров: 61.500 мс (base R) и 28.500 мс (dplyr) на вызов.


Результаты опроса описывают ответивших участников. Сравнение времени относится к выбранной операции, таблице и среде запуска.